[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahirtonlopes/AI-and-Robotics-Trends/blob/main/Notebooks/Demo5_Agrupamento_KDD99.ipynb)

# Demo 5 - Demonstração - Aprendizagem Não Supervisionada: agrupando tráfego de rede

### Cybersecurity MBA - Artificial Intelligence & Robotics Trends (SEG)

### Os ataques formam grupos sozinhos? K-Means e detecção de anomalias SEM usar rótulo

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

**A base:** KDD Cup 99, o **clássico** dos problemas de detecção de intrusão sem rótulo (tráfego simulado do programa de avaliação da DARPA de 1998). Foi usada no artigo pioneiro de agrupamento para detecção de intrusão sem dados rotulados (Portnoy, Eskin e Stolfo, 2001). **Cuidado:** é uma base de 1999, com problemas conhecidos (muitos registros repetidos, Tavallaee et al., 2009) e não representa uma rede de hoje. Aqui ela serve para aprender a **técnica**; na Demo 4 usamos uma base mais moderna (UNSW-NB15).

## 1. Carregar os dados

O subconjunto **SA** tem todo o tráfego normal e poucos ataques (anomalias raras, como num SOC de verdade, mas ainda mais frequentes que na vida real).

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import fetch_kddcup99
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score, roc_auc_score, precision_score, recall_score

# KDD Cup 99, subconjunto 'SA' (todo o tráfego normal + uma fração pequena de ataques), versão de 10%.
# O scikit-learn baixa o arquivo sozinho na primeira vez.
X, y = fetch_kddcup99(subset="SA", percent10=True, return_X_y=True, as_frame=True, random_state=0)   # random_state fixa a amostra de ataques do subconjunto SA
y = y.astype(str).str.replace("b'", "").str.replace("'", "").str.strip(".")       # rótulos (só para CONFERIR no fim)
ataque = (y != "normal").astype(int).values
print(f"{len(X):,} conexões | {X.shape[1]} características | ataques: {ataque.mean():.1%}")
print(y.value_counts().head(6).to_dict())

## 2. Preparar (sem usar o rótulo)

In [ ]:
# Preparação: números em escala logarítmica (bytes variam de 0 a milhões) e padronização.
# NÃO usamos o rótulo em nenhum passo do agrupamento.
cat = ["protocol_type", "service", "flag"]
Xn = np.log1p(X.drop(columns=cat).astype(float).clip(lower=0))
proto = pd.get_dummies(X["protocol_type"].astype(str).str.replace("b'", "").str.replace("'", "")).astype(float)
Z = StandardScaler().fit_transform(pd.concat([Xn, proto], axis=1))
print("matriz para o K-Means:", Z.shape)

## 3. Quantos grupos? 

O K-Means **sempre** devolve k grupos. A inércia e a silhouette ajudam, mas **não decidem**: o objetivo decide.

In [ ]:
# Quantos grupos? Inércia (cotovelo) e silhouette.
print("k | inércia  | silhouette")
for k in (2, 3, 5, 8, 10, 15, 20):
    m = KMeans(n_clusters=k, n_init=5, random_state=0).fit(Z)
    sil = silhouette_score(Z, m.labels_, sample_size=5000, random_state=0)
    print(f"{k:2d} | {m.inertia_:9.0f} | {sil:.3f}")

**Leitura:** a silhouette é maior com k = 2 (um grupo grande e um pequeno): é separação 'barata', não útil. Para achar famílias de tráfego, precisamos de mais grupos. Escolheremos k = 20 e veremos o que os grupos significam.

In [ ]:
K = 20
km = KMeans(n_clusters=K, n_init=5, random_state=0).fit(Z)
lab = km.labels_
tam = pd.Series(lab).value_counts().sort_index()
print("tamanho dos grupos (do menor para o maior):", sorted(tam.tolist()))

## 4. Desenhar os grupos em 2D (PCA)

In [ ]:
# Desenhar 100 mil conexões em 2D com PCA: cada ponto é uma conexão; a cor é o GRUPO (sem rótulo).
pca = PCA(n_components=2, random_state=0).fit(Z)
P = pca.transform(Z)
rng = np.random.default_rng(0); amostra = rng.choice(len(P), 15000, replace=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
ax[0].scatter(P[amostra, 0], P[amostra, 1], c=lab[amostra], cmap="tab20", s=4); ax[0].set_title("Cor = grupo do K-Means (sem usar rótulo)")
cores = np.where(ataque[amostra] == 1, "#EB005A", "#9AA0A6")
ax[1].scatter(P[amostra, 0], P[amostra, 1], c=cores, s=4); ax[1].set_title("Cor = rótulo real (rosa = ataque)  [só para conferir]")
for a in ax: a.set_xlabel("PC1"); a.set_ylabel("PC2")
plt.tight_layout(); plt.show()
print(f"variância explicada por PC1 e PC2: {pca.explained_variance_ratio_.sum():.1%}")

## 5. Revelar o rótulo: os grupos têm significado?

Só agora olhamos a verdade. Note: os ataques volumétricos (`smurf` e `neptune`, tipos de DoS) formam grupos quase puros; os outros ataques são raros e, em geral, se misturam ao tráfego normal.

In [ ]:
# AGORA revelamos o rótulo: de que são feitos os grupos?
tipos = y.where(y.isin(["normal", "smurf", "neptune"]), "outros ataques")
comp = pd.crosstab(lab, tipos)[["normal", "smurf", "neptune", "outros ataques"]]
comp["total"] = comp.sum(axis=1)
comp["% ataque"] = ((comp["total"] - comp["normal"]) / comp["total"] * 100).round(1)
display(comp.sort_values("% ataque", ascending=False)) if "display" in globals() else print(comp.sort_values("% ataque", ascending=False))
pureza = np.mean([max(c, t - c) / t for c, t in zip(comp["normal"], comp["total"])])
pureza_pond = sum(max(c, t - c) for c, t in zip(comp["normal"], comp["total"])) / len(Z)
print(f"pureza global (normal x ataque): {pureza_pond:.1%}  |  referência: prever tudo como normal acerta {1 - ataque.mean():.1%}")
print("Cuidado: a pureza global é alta só porque quase tudo é normal. O que interessa são os grupos de ATAQUE (% ataque alta).")

## 6. A heurística clássica: 'grupo pequeno = ataque'

Portnoy et al. (2001) assumem que o tráfego normal é a imensa maioria. Vejamos se isso funciona aqui.

In [ ]:
# Heurística clássica (Portnoy, Eskin e Stolfo, 2001): os ataques são RAROS, então GRUPOS PEQUENOS = anômalos.
tam = np.bincount(lab); print("limite | grupos pequenos | precision | recall")
for lim in (0.01, 0.02, 0.05):
    pequenos = [c for c in range(K) if tam[c] / len(Z) < lim]
    pred = np.isin(lab, pequenos).astype(int)
    print(f"{lim:5.0%} | {len(pequenos):15d} | {precision_score(ataque, pred, zero_division=0):9.2f} | {recall_score(ataque, pred):.2f}")

**Leitura:** os ataques mais **comuns** nesta base (os DoS volumétricos) formam grupos **grandes e densos**, então a regra 'raro = ataque' os perde; e muitos grupos pequenos são tráfego normal incomum. A heurística pega pouco e erra muito: recall e precision caminham em sentidos opostos conforme o limite.

## 7. Detecção de anomalias sem rótulo: Isolation Forest

In [ ]:
# Detecção de anomalias SEM rótulo: Isolation Forest (pontos fáceis de isolar = estranhos).
iso = IsolationForest(n_estimators=100, random_state=0).fit(Z)
score = -iso.score_samples(Z)                      # maior = mais estranho
print(f"AUC (ranking de estranheza x ataque): {roc_auc_score(ataque, score):.3f}")
corte = np.quantile(score, 0.95); pred = (score >= corte).astype(int)
print(f"alertar nos 5% mais estranhos: precision={precision_score(ataque, pred):.2f} recall={recall_score(ataque, pred):.2f}")
# Variante: treinar só com tráfego que CONHECEMOS como normal (semi-supervisionado)
iso2 = IsolationForest(n_estimators=100, random_state=0).fit(Z[ataque == 0])
print(f"AUC treinando só em normal: {roc_auc_score(ataque, -iso2.score_samples(Z)):.3f}")

**Leitura:** o ranking de estranheza separa ataque de normal bem melhor que o acaso (AUC alto), mas **alertar nos 5% mais estranhos** gera muito falso positivo (precision baixa). Treinar só com tráfego **conhecidamente normal** melhora o ranking: é o caminho semi-supervisionado que se usa na prática.

## O que concluir

- Agrupamento **achou estrutura** (famílias de tráfego, inclusive dois tipos de DoS quase puros) **sem ver o rótulo**.
- Mas **não achou 'o ataque'** de graça: ataques raros ficam misturados ao normal, e 'raro' não é sinônimo de 'ataque'.
- Métricas internas (silhouette) **não** escolhem k: quem escolhe é o objetivo.
- Em um SOC: use agrupamento para **explorar e priorizar**, e anomalia para **gerar suspeitas**. Um analista (ou um modelo supervisionado da Demo 4) decide.
- Lembre a Aula 2: com prevalência baixa, a **precision** de qualquer alerta cai.

## Para praticar

1. Rode com `K = 5`, `10` e `30`. Quais famílias de ataque aparecem como grupos puros em cada k?
2. Remova as colunas `src_bytes` e `dst_bytes`. O que acontece com os grupos de `smurf`?
3. Padronização: rode **sem** `StandardScaler` e compare. Por que importa a escala?
4. Use o subconjunto `subset='SF'` (menos de 1% de ataques) e refaça a Isolation Forest. A precision muda?
5. **Discussão:** em que ponto do fluxo de um SOC você colocaria o agrupamento? Quem nomeia cada grupo?
6. **Opcional:** pense em um caso de agrupamento com dados de pessoas (por exemplo, crimes por região). O que é parecido e o que muda quando os dados são de pessoas, e não de pacotes?